In [1]:
# STEP 1: Load dataset and describe script
# Description:
# This script calculates the number of possible PV panels that can fit on ONE side of each roof.
# It uses ROOF_LENGTH and ROOF_WIDTH from the dataset and assumes:
# - Panels are placed in landscape orientation (1.8m along roof length, 1.134m along roof width)
# - Only 85% of roof dimensions are usable
# - 4 panels are subtracted to allow for solar hot water
# - Final result is rounded down to the nearest whole number

import pandas as pd
import numpy as np

# File paths
input_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_roof_area_update.csv"
output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_pv_update.csv"

# Load dataset
df = pd.read_csv(input_path)

# Preview
print(df.head())

   BUILDING_REFERENCE_NUMBER  OSG_REFERENCE_NUMBER              ADDRESS1  \
0                 1001829067           122009933.0  19 CULCREUCH AVENUE    
1                 1001951858           122010025.0             GLENVIEW    
2                 1001829071           122009868.0  22 CULCREUCH AVENUE    
3                 1234761001           122009968.0    1 MENZIES TERRACE    
4                 1001991633           122009892.0       50 MAIN STREET    

  ADDRESS2  ADDRESS3 POSTCODE   LATITUDE  LONGITUDE INSPECTION_DATE  \
0  FINTRY   GLASGOW   G63 0YB  56.055692  -4.223337         9/29/25   
1  FINTRY   GLASGOW   G63 0XJ  56.052824  -4.220640         9/19/25   
2  FINTRY   GLASGOW   G63 0YB  56.055503  -4.223691         7/29/25   
3  FINTRY   GLASGOW   G63 0YJ  56.058427  -4.224838         7/22/25   
4  FINTRY   GLASGOW   G63 0XF  56.054738  -4.228307         7/17/25   

         TYPE_OF_ASSESSMENT  ...   ROOF_AREA  USABLE_ROOF_LENGTH  \
0  RdSAP, existing dwelling  ...   29.698485    

In [2]:
# STEP 2: Define panel dimensions and usable roof area
# Description: Applies 85% usability factor to roof dimensions

PANEL_LENGTH = 1.8     # meters (aligned with roof length)
PANEL_WIDTH = 1.134    # meters (aligned with roof width)

# Usable dimensions
df["USABLE_ROOF_LENGTH"] = df["ROOF_LENGTH"] * 0.85
df["USABLE_ROOF_WIDTH"] = df["ROOF_WIDTH"] * 0.85

print(df[["ROOF_LENGTH", "ROOF_WIDTH", "USABLE_ROOF_LENGTH", "USABLE_ROOF_WIDTH"]].head())

   ROOF_LENGTH  ROOF_WIDTH  USABLE_ROOF_LENGTH  USABLE_ROOF_WIDTH
0     4.582576    3.240370            3.895189           2.754315
1     9.721111    4.582576            8.262944           3.895189
2     5.830952    4.123106            4.956309           3.504640
3     9.759611    4.600725            8.295669           3.910616
4    12.489996    7.772429           10.616497           6.606565


In [3]:
# STEP 3: Calculate number of panels along each dimension
# Description: Determines how many panels fit along length and width (rounded down)

df["PANELS_ALONG_LENGTH"] = np.floor(df["USABLE_ROOF_LENGTH"] / PANEL_LENGTH)
df["PANELS_ALONG_WIDTH"] = np.floor(df["USABLE_ROOF_WIDTH"] / PANEL_WIDTH)

print(df[["PANELS_ALONG_LENGTH", "PANELS_ALONG_WIDTH"]].head())

   PANELS_ALONG_LENGTH  PANELS_ALONG_WIDTH
0                  2.0                 2.0
1                  4.0                 3.0
2                  2.0                 3.0
3                  4.0                 3.0
4                  5.0                 5.0


In [4]:
# STEP 4: Calculate total panels and subtract allowance
# Description: Total panels = grid fit minus 4 panels for solar hot water

df["POSSIBLE_PV_PANELS"] = (
    (df["PANELS_ALONG_LENGTH"] * df["PANELS_ALONG_WIDTH"]) - 4
)

# Ensure no negative values
df["POSSIBLE_PV_PANELS"] = df["POSSIBLE_PV_PANELS"].clip(lower=0)

# Convert to integer
df["POSSIBLE_PV_PANELS"] = df["POSSIBLE_PV_PANELS"].astype(int)

print(df[["PANELS_ALONG_LENGTH", "PANELS_ALONG_WIDTH", "POSSIBLE_PV_PANELS"]].head())

   PANELS_ALONG_LENGTH  PANELS_ALONG_WIDTH  POSSIBLE_PV_PANELS
0                  2.0                 2.0                   0
1                  4.0                 3.0                   8
2                  2.0                 3.0                   2
3                  4.0                 3.0                   8
4                  5.0                 5.0                  21


In [5]:
# STEP 5: Save updated dataset
# Description: Writes the dataset with PV calculations to CSV

df.to_csv(output_path, index=False)

print(f"File saved to: {output_path}")

File saved to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_pv_update.csv
